In [27]:
import pandas as pd
import altair as alt

In [28]:
#Carga de datos
df = pd.read_csv('../../sources/data-javier/dataset_olympics.csv')

In [29]:
#Agrupamiento de datos por año y país contando ID únicos de atletas
df_grupo = df.groupby(['Year', 'NOC'])['ID'].nunique().reset_index()
df_grupo.rename(columns = {'ID': 'Participantes'}, inplace = True)

In [32]:
#Filtrado por juegos de Verano y distinto de 1906 de Atenas(un juego intermedio que no es considerado oficial)
df_verano = df[(df['Season'] == 'Summer') & (df['Year'] != 1906)] 

In [64]:
#Obtener la ciudad de realización de los JJOO en cada año
df_ciudades = df_verano[['Year', 'City']].drop_duplicates().sort_values('Year')
#En 1956 se realizaron los JJOO en 2 sedes, Melbourne y Estocolmo,se quita Estocolmo para unificarlo
df_ciudades = df_ciudades[~((df_ciudades['Year'] ==1956) & (df_ciudades['City'] == 'Stockholm'))]
diccionario_ciudades = dict(zip(df_ciudades['Year'], df_ciudades['City']))

In [65]:
#Añadir juegos cancelados
diccionario_ciudades[1916] = 'Berlin (Cancelado)'
diccionario_ciudades[1940] = 'Tokyo (Cancelado)'
diccionario_ciudades[1944] = 'London (Cancelado)'

In [66]:
#Asignar NOC al nombre de país más común
noc_nombres = df_verano.groupby(['NOC', 'Team']).size().reset_index(name='count')
noc_nombres = noc_nombres.loc[noc_nombres.groupby('NOC')['count'].idxmax()]
diccionario_nombres_paises = dict(zip(noc_nombres['NOC'], noc_nombres['Team']))

In [67]:
#Agrupar atletas por NOC y año
df_grupo = df_verano.groupby(['Year', 'NOC'])['ID'].nunique().reset_index()
df_grupo.rename(columns={'ID': 'Participantes'}, inplace=True)

In [68]:
#Obtener los 10 países con mayor participación histórica
top_paises = df_grupo.groupby('NOC')['Participantes'].sum().nlargest(10).index

In [69]:
#Agrupar los países fuera del top 10 en otros
df_grupo['Categoria_pais'] = df_grupo['NOC'].apply(lambda x: diccionario_nombres_paises.get(x, x) if x in top_paises else 'Otros')

In [70]:
#Volver a agrupar los top 10 países junto con los que están fuera del top 10
df_agrupado_final = df_grupo.groupby(['Year', 'Categoria_pais'])['Participantes'].sum().reset_index()

In [71]:
#Meter años donde no hubo juegos para que no aparezcan participantes
anos_completos = list(range(1896, 2017, 4))
categorias = df_agrupado_final['Categoria_pais'].unique()
datos_completos = []
for ano in anos_completos:
    for cat in categorias:
        match = df_agrupado_final[(df_agrupado_final['Year'] == ano) & (df_agrupado_final['Categoria_pais'] == cat)]
        if not (match.empty):
            participantes = match['Participantes'].values[0]
        else:
            participantes = 0 #Cuando no hubo participantes
        ciudad = diccionario_ciudades.get(ano, "Desconocida")
        label_eje = f"{ano} - {ciudad}"
        datos_completos.append({
            'Year': ano,
            'Categoria_pais': cat,
            'Participantes': participantes,
            'Eje_X_Label': label_eje
        })
df_grafico = pd.DataFrame(datos_completos)

In [72]:
#Definir orden para que "Otros" quedfe último en el agrupamiento visual
orden_paises = [diccionario_nombres_paises.get(noc, noc) for noc in top_paises] + ['Otros']

In [73]:
#Generando el gráfico en Altair 
grafico = alt.Chart(df_grafico).mark_area(opacity=0.8).encode(
    x = alt.X('Eje_X_Label:N', sort = alt.EncodingSortField(field = 'Year', order = 'ascending'), title = 'Año de los juegos y Sede', axis = alt.Axis(labelAngle=-45, labelOverlap = False)),
    y = alt.Y('Participantes:Q', stack = 'center', title = 'Cantidad de Atletas', axis = None),
    color = alt.Color('Categoria_pais:N', sort = orden_paises, title = 'Países en el top 10 y otros', scale = alt.Scale(scheme = 'tableau20')),
    order = alt.Order('color_Categoria_pais_sort_index:Q', sort = 'descending'),
    tooltip = [
        alt.Tooltip('Categoria_pais:N', title = 'País'),
        alt.Tooltip('Eje_X_Label:N', title = 'Año'),
        alt.Tooltip('Participantes:Q', title = 'Atletas')
    ]
).properties(
    title = 'Evolución de Participaciones por País en los Juegos Olímpicos',
    width = 750,
    height = 400
)

In [74]:
grafico

alt.Chart(...)